In [1]:
# various import statements
import os
from dotenv import load_dotenv
import pandas as pd
import gspread
from google.oauth2.service_account import Credentials
import matplotlib.pyplot as plt 
import seaborn as sbn
from scipy import stats
import plotly.express as px
import plotly.graph_objects as go
from great_tables import GT, md, html, style, loc

## Initial data load and score calculation

In [2]:
# load in data from google sheet where my scores are stored
load_dotenv()
scope = ['https://www.googleapis.com/auth/drive']
jsonpath = os.environ.get('GOOGLE_APPLICATION_CREDENTIALS')
spreadsheet_id = '1suD1XqF5Dr5PdR3ggxxIyOo2_6vYakOnH1Afha72SR0'
creds = Credentials.from_service_account_file(jsonpath, scopes=scope)
gc = gspread.authorize(creds)
spreadsheet = gc.open_by_key(spreadsheet_id)
worksheet = spreadsheet.sheet1

In [3]:
data = worksheet.get_all_values()
score_df = pd.DataFrame(data[1:], columns=data[0])
score_df.head(10)

,Restaurant Name,City,Neighborhood,Cuisine,Sub Cuisine,Visit Date,Ingredient Quality,Execution,Personality,Price to Quality,Price to Portion,Sensory,Design,Vibe,Pacing,Staff
0,Akahoshi Ramen,Chicago,Logan Square,Japanese,Ramen,3/20/2026,8,9,8,8,7,7,5,7,8,7
1,Cabra,Chicago,Fulton Market,Peruvian,,3/21/2026,6,7,7,6,6,4,6,7,5,5
2,Momotaro,Chicago,Fulton Market,Japanese,,3/27/2026,8,8,7,7,4,6,9,8,7,8
3,Mi Tocaya Antojeria,Chicago,Logan Square,Mexican,,3/28/2026,9,9,10,9,10,7,8,9,6,7
4,Sully's House,Chicago,Lincoln Park,American,Bar,3/29/2026,5,5,3,4,6,6,6,5,6,7
5,Aberdeen Tap,Chicago,West Town,American,Bar,4/2/2026,5,6,5,5,5,6,6,7,5,5
6,Tacqueria Chingon,Chicago,Fulton Market,Mexican,,4/10/2026,7,7,8,7,6,5,7,7,6,5
7,Oakville Grill & Cellar,Chicago,Fulton Market,American,California,4/12/2026,7,6,5,5,6,5,7,7,6,7


In [4]:
# score weighting definition

# food score weighting
ingredient_qual_weight = 0.3
execution_weight = 0.4
personality_weight = 0.3

# value score weighting
price_to_quality = 0.7
price_to_portion = 0.3

# atmosphere weighting
sensory_weight = 0.3
design_weight = 0.3
vibe_weight = 0.4

# hospitality weighting
pacing_weight = 0.3
staff_weight = 0.7

# overall score weighting
food_weight = 0.4
value_weight = 0.2
atmosphere_weight = 0.2
hospitality_weight = 0.2

# define variable to grab just score columns
cat_score_cols = [
    'Ingredient Quality', 
    'Execution', 
    'Personality', 
    'Price to Quality', 
    'Price to Portion', 
    'Sensory', 
    'Design', 
    'Vibe', 
    'Pacing', 
    'Staff'
]

# ensure each score col is int
score_df[cat_score_cols] = score_df[cat_score_cols].astype(int)

In [5]:
# score calculation

score_df['Food Score'] = (
    score_df['Ingredient Quality'] * ingredient_qual_weight +
    score_df['Execution'] * execution_weight +
    score_df['Personality'] * personality_weight
).round(1)

score_df['Value Score'] = (
    score_df['Price to Quality'] * price_to_quality +
    score_df['Price to Portion'] * price_to_portion
).round(1)

score_df['Atmosphere Score'] = (
    score_df['Sensory'] * sensory_weight +
    score_df['Design'] * design_weight +
    score_df['Vibe'] * vibe_weight
).round(1)

score_df['Hospitality Score'] = (
    score_df['Pacing'] * pacing_weight +
    score_df['Staff'] * staff_weight
).round(1)

# overall score calculation
score_df['Overall Score'] = (
    score_df['Food Score'] * food_weight +
    score_df['Value Score'] * value_weight +
    score_df['Atmosphere Score'] * atmosphere_weight +
    score_df['Hospitality Score'] * hospitality_weight
).round(1)

score_df.head()

,Restaurant Name,City,Neighborhood,Cuisine,Sub Cuisine,Visit Date,Ingredient Quality,Execution,Personality,Price to Quality,...,Sensory,Design,Vibe,Pacing,Staff,Food Score,Value Score,Atmosphere Score,Hospitality Score,Overall Score
0,Akahoshi Ramen,Chicago,Logan Square,Japanese,Ramen,3/20/2026,8,9,8,8,...,7,5,7,8,7,8.4,7.7,6.4,7.3,7.6
1,Cabra,Chicago,Fulton Market,Peruvian,,3/21/2026,6,7,7,6,...,4,6,7,5,5,6.7,6.0,5.8,5.0,6.0
2,Momotaro,Chicago,Fulton Market,Japanese,,3/27/2026,8,8,7,7,...,6,9,8,7,8,7.7,6.1,7.7,7.7,7.4
3,Mi Tocaya Antojeria,Chicago,Logan Square,Mexican,,3/28/2026,9,9,10,9,...,7,8,9,6,7,9.3,9.3,8.1,6.7,8.5
4,Sully's House,Chicago,Lincoln Park,American,Bar,3/29/2026,5,5,3,4,...,6,6,5,6,7,4.4,4.6,5.6,6.7,5.1


In [10]:
# update this line to generate associated charts
curr_review = 'Oakville Grill & Cellar'

In [11]:

# grab total sample
total_restaurants = len(score_df)

# define the higher level category columns
high_level_cols = [
    'Food Score', 'Value Score', 'Atmosphere Score', 'Hospitality Score', 'Overall Score'
]
# calculate the rank for each column across the entire dataset
ranks = score_df[high_level_cols].rank(ascending=False, method='min')
total_restaurants = len(score_df)

# extract the scores and ranks purely for the current review
curr_review_row = score_df[score_df['Restaurant Name'] == curr_review].iloc[0]
curr_review_ranks = ranks.loc[curr_review_row.name]

# create a DataFrame to hold the table data
summary_data = {
    'Category': high_level_cols,
    'Score': [curr_review_row[col] for col in high_level_cols],
    'Rank': [f"{int(curr_review_ranks[col])} out of {total_restaurants}" for col in high_level_cols]
}
summary_df = pd.DataFrame(summary_data)

# build and style the great_tables object
summary_table = (
    GT(summary_df)
    .tab_header(
        title="Category Scores & Rankings",
        subtitle=curr_review
    )
    .cols_align(align="center", columns=['Score', 'Rank'])
    .tab_style(
        style=style.text(weight="bold"),
        locations=loc.body(rows=[4]) # Bold the 5th row ('Overall Score')
    )
)
# save the table as png for inclusion in article
# summary_table.save(f"{curr_review} rankings.png")

## Review vs Cuisine Spider

In [12]:
# get score values for current selection only
curr_review_scores = score_df.loc[score_df['Restaurant Name'] == curr_review, cat_score_cols].iloc[0]

# create list of just current review values for spider chart
review_vals = curr_review_scores.values.tolist()
review_vals.append(review_vals[0])

# get average score values for current selection cuisine
curr_cuisine = score_df.loc[score_df['Restaurant Name'] == curr_review, 'Cuisine'].values[0]
cuisine_avgs = score_df.groupby('Cuisine')[cat_score_cols].mean()
curr_cuisine_avg = cuisine_avgs.loc[curr_cuisine]

# create list for shared categories
categories = curr_cuisine_avg.index.tolist()
categories.append(categories[0])

# create list of scores for 
avg_vals = curr_cuisine_avg.values.tolist()
avg_vals.append(avg_vals[0])

fig = go.Figure()

fig.add_trace(go.Scatterpolar(
    r=avg_vals,
    theta=categories,
    fill='toself',
    name=f'{curr_cuisine} Average',
    fillcolor='rgba(150, 150, 150, 0.3)',
    line=dict(color='gray')
))

fig.add_trace(go.Scatterpolar(
    r=review_vals,
    theta=categories,
    fill='toself',
    name=curr_review,
    fillcolor='rgba(255, 0, 0, 0.2)',
    line=dict(color='red', width=2)
))

# html styled title to act as legend instead of separate legend
styled_title = f"<span style='color: red; font-weight: bold;'>{curr_review}</span> vs. {curr_cuisine} Cuisine Averages"

fig.update_layout(
    title=styled_title,
    title_x=0.5,
    width=600,
    height=500,
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, 10]
        )
    ),
    showlegend=False
)
fig.show()


## Review vs City Avg Spider

In [13]:

# calculate the overall average scores across all restaurants
overall_avg = score_df[cat_score_cols].mean()

# setup variables to close the polygons
categories = overall_avg.index.tolist()
categories.append(categories[0])
overall_avg_vals = overall_avg.values.tolist()
overall_avg_vals.append(overall_avg_vals[0])

# using the same review score values mapped from earlier
review_vals = curr_review_scores.values.tolist()
review_vals.append(review_vals[0])

# build the chart
fig = go.Figure()

fig.add_trace(go.Scatterpolar(
    r=overall_avg_vals,
    theta=categories,
    fill='toself',
    name=f'Average',
    fillcolor='rgba(150, 150, 150, 0.3)',
    line=dict(color='gray')
))

fig.add_trace(go.Scatterpolar(
    r=review_vals,
    theta=categories,
    fill='toself',
    name=curr_review,
    fillcolor='rgba(255, 0, 0, 0.2)',
    line=dict(color='red', width=2) 
))

styled_title = f"<span style='color: red; font-weight: bold;'>{curr_review}</span> vs. Average"

fig.update_layout(
    title=styled_title,
    title_x=0.5,
    width=600,
    height=500,
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, 10]
        )
    ),
    showlegend=False
)

fig.show()
